In [1]:
import glob
import polars as pl

# Find all Wikimedia Structured Wikipedia Parquet files
files = glob.glob(
    "/kaggle/input/datasets/wikimedia-foundation/wikipedia-structured-contents/enwiki/data/*.parquet"
)

print("Parquet files found:", len(files))

Parquet files found: 265


In [2]:
# Read the first Wikimedia Structured Wikipedia file

df = pl.read_parquet(files[0])

print("Rows:", df.height)
print("\nColumns:")
print(df.columns)

Rows: 25000

Columns:
['abstract', 'additional_entities', 'date_created', 'date_modified', 'description', 'event', 'identifier', 'image', 'in_language', 'infoboxes', 'is_part_of', 'license', 'main_entity', 'name', 'references', 'sections', 'tables', 'url', 'version']


In [3]:
# Inspect the fields needed for WikiWeak Article Finder

print("Sample articles:")
print(df.select(["name", "url"]).head(5))

print("\nData types of important fields:")
print(
    df.select([
        "name",
        "sections",
        "infoboxes",
        "image",
        "references",
        "version"
    ]).schema
)

Sample articles:
shape: (5, 2)
┌─────────────────────────────┬─────────────────────────────────┐
│ name                        ┆ url                             │
│ ---                         ┆ ---                             │
│ str                         ┆ str                             │
╞═════════════════════════════╪═════════════════════════════════╡
│ Helmingham Dell             ┆ https://en.wikipedia.org/wiki/… │
│ Klaipėda Free Economic Zone ┆ https://en.wikipedia.org/wiki/… │
│ Swedish for immigrants      ┆ https://en.wikipedia.org/wiki/… │
│ Tiquadra syntripta          ┆ https://en.wikipedia.org/wiki/… │
│ The Mighty Jingles          ┆ https://en.wikipedia.org/wiki/… │
└─────────────────────────────┴─────────────────────────────────┘

Data types of important fields:
Schema({'name': String, 'sections': String, 'infoboxes': String, 'image': Struct({'content_url': String, 'height': Int64, 'width': Int64}), 'references': List(Struct({'identifier': String, 'metadata': String, '

In [4]:
# Calculate the five factors for WikiWeak Article Finder

def count_items(value):
    if value is None:
        return 0

    if isinstance(value, list):
        return len(value)

    if isinstance(value, dict):
        return len(value)

    if isinstance(value, str):
        value = value.strip()

        if not value:
            return 0

        # Count non-empty lines for serialized structured data
        return len([line for line in value.splitlines() if line.strip()])

    return 0


def count_image(value):
    # The Wikimedia dataset stores the image field as a single image struct.
    return 0 if value is None else 1


results = df.select([
    "name",
    "url",

    # Article length supplied by Wikimedia
    pl.col("version")
      .struct.field("number_of_characters")
      .alias("character_count"),

    # Number of sections
    pl.col("sections")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("section_count"),

    # Infobox information
    pl.col("infoboxes")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("infobox_field_count"),

    # Image presence
    pl.col("image")
      .map_elements(count_image, return_dtype=pl.Int64)
      .alias("image_count"),

    # References
    pl.col("references")
      .map_elements(count_items, return_dtype=pl.Int64)
      .alias("reference_count")
])

print(results.head())

shape: (5, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ name         ┆ url         ┆ character_c ┆ section_cou ┆ infobox_fie ┆ image_count ┆ reference_c │
│ ---          ┆ ---         ┆ ount        ┆ nt          ┆ ld_count    ┆ ---         ┆ ount        │
│ str          ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ i64         ┆ ---         │
│              ┆             ┆ i64         ┆ i64         ┆ i64         ┆             ┆ i64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Helmingham   ┆ https://en. ┆ 3213        ┆ 1           ┆ 1           ┆ 1           ┆ 0           │
│ Dell         ┆ wikipedia.o ┆             ┆             ┆             ┆             ┆             │
│              ┆ rg/wiki/…   ┆             ┆             ┆             ┆             ┆             │
│ Klaipėda     ┆ https://en. ┆ 8861        ┆ 1           ┆ 1           ┆ 1   

In [5]:
 # Replace missing values with 0

results = results.with_columns([
    pl.col("character_count").fill_null(0),
    pl.col("section_count").fill_null(0),
    pl.col("infobox_field_count").fill_null(0),
    pl.col("image_count").fill_null(0),
    pl.col("reference_count").fill_null(0)
])

print(
    results.select([
        "name",
        "character_count",
        "section_count",
        "infobox_field_count",
        "image_count",
        "reference_count"
    ]).head(10)
)

shape: (10, 6)
┌─────────────────┬────────────────┬───────────────┬────────────────┬─────────────┬────────────────┐
│ name            ┆ character_coun ┆ section_count ┆ infobox_field_ ┆ image_count ┆ reference_coun │
│ ---             ┆ t              ┆ ---           ┆ count          ┆ ---         ┆ t              │
│ str             ┆ ---            ┆ i64           ┆ ---            ┆ i64         ┆ ---            │
│                 ┆ i64            ┆               ┆ i64            ┆             ┆ i64            │
╞═════════════════╪════════════════╪═══════════════╪════════════════╪═════════════╪════════════════╡
│ Helmingham Dell ┆ 3213           ┆ 1             ┆ 1              ┆ 1           ┆ 0              │
│ Klaipėda Free   ┆ 8861           ┆ 1             ┆ 1              ┆ 1           ┆ 0              │
│ Economic Zone   ┆                ┆               ┆                ┆             ┆                │
│ Swedish for     ┆ 10226          ┆ 1             ┆ 0              ┆ 1     

In [6]:
# Calculate a combined Content-Richness Score
# Higher score = more content-rich
# Lower score = relatively limited content

factors = {
    "character_count": 0.40,
    "section_count": 0.20,
    "reference_count": 0.20,
    "infobox_field_count": 0.10,
    "image_count": 0.10
}

# Convert each factor into a percentile score
for column in factors:
    results = results.with_columns(
        (
            pl.col(column).rank(method="average")
            / results.height
            * 100
        ).alias(f"{column}_percentile")
    )

# Weighted combination of all five factors
results = results.with_columns(
    (
        pl.col("character_count_percentile") * 0.40 +
        pl.col("section_count_percentile") * 0.20 +
        pl.col("reference_count_percentile") * 0.20 +
        pl.col("infobox_field_count_percentile") * 0.10 +
        pl.col("image_count_percentile") * 0.10
    ).alias("content_richness_score")
)

print(
    results.select([
        "name",
        "character_count",
        "section_count",
        "image_count",
        "infobox_field_count",
        "reference_count",
        "content_richness_score"
    ]).head(10)
)

shape: (10, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ name         ┆ character_c ┆ section_cou ┆ image_count ┆ infobox_fie ┆ reference_c ┆ content_ric │
│ ---          ┆ ount        ┆ nt          ┆ ---         ┆ ld_count    ┆ ount        ┆ hness_score │
│ str          ┆ ---         ┆ ---         ┆ i64         ┆ ---         ┆ ---         ┆ ---         │
│              ┆ i64         ┆ i64         ┆             ┆ i64         ┆ i64         ┆ f64         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Helmingham   ┆ 3213        ┆ 1           ┆ 1           ┆ 1           ┆ 0           ┆ 44.4246     │
│ Dell         ┆             ┆             ┆             ┆             ┆             ┆             │
│ Klaipėda     ┆ 8861        ┆ 1           ┆ 1           ┆ 1           ┆ 0           ┆ 60.1678     │
│ Free         ┆             ┆             ┆             ┆             ┆    

In [7]:
# Identify relatively limited-content articles
# We use the lowest 5% of the combined score.

threshold = results.select(
    pl.col("content_richness_score").quantile(0.05)
).item()

results = results.with_columns(
    (
        pl.col("content_richness_score") <= threshold
    ).alias("limited_content")
)

print(f"Content-richness threshold: {threshold:.2f}")

print(
    "Limited-content articles:",
    results.filter(
        pl.col("limited_content")
    ).height
)

Content-richness threshold: 29.90
Limited-content articles: 1251


In [8]:
# Rank articles from lowest to highest Content-Richness Score

results = results.sort("content_richness_score")

results = results.with_row_index(
    "rank",
    offset=1
)

print(
    results.select([
        "rank",
        "name",
        "content_richness_score",
        "character_count",
        "section_count",
        "image_count",
        "infobox_field_count",
        "reference_count",
        "limited_content"
    ]).head(10)
)

shape: (10, 9)
┌──────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ rank ┆ name       ┆ content_ri ┆ character_ ┆ … ┆ image_coun ┆ infobox_f ┆ reference ┆ limited_c │
│ ---  ┆ ---        ┆ chness_sco ┆ count      ┆   ┆ t          ┆ ield_coun ┆ _count    ┆ ontent    │
│ u32  ┆ str        ┆ re         ┆ ---        ┆   ┆ ---        ┆ t         ┆ ---       ┆ ---       │
│      ┆            ┆ ---        ┆ i64        ┆   ┆ i64        ┆ ---       ┆ i64       ┆ bool      │
│      ┆            ┆ f64        ┆            ┆   ┆            ┆ i64       ┆           ┆           │
╞══════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 1    ┆ Arjun Ray  ┆ 23.875     ┆ 99         ┆ … ┆ 0          ┆ 0         ┆ 0         ┆ true      │
│ 2    ┆ Polish     ┆ 23.8766    ┆ 117        ┆ … ┆ 0          ┆ 0         ┆ 0         ┆ true      │
│      ┆ Uplanders  ┆            ┆            ┆   ┆            ┆           ┆

In [9]:
# Select the lowest-scoring limited-content article

limited_article = (
    results
    .filter(pl.col("limited_content") == True)
    .sort("content_richness_score")
    .row(0, named=True)
)

print("=" * 55)
print("             WIKIWEAK ARTICLE ANALYSIS")
print("=" * 55)

print("Article:", limited_article["name"])
print("Character Count:", limited_article["character_count"])
print("Sections:", limited_article["section_count"])
print("Images:", limited_article["image_count"])
print("Infobox Fields:", limited_article["infobox_field_count"])
print("References:", limited_article["reference_count"])

print(
    "Content-Richness Score:",
    round(limited_article["content_richness_score"], 2)
)

print("\nAssessment:")
print(
    "This article has relatively limited measured content "
    "based on the combined analysis of article length, sections, "
    "images, infobox information, and references."
)

print("\nNote:")
print(
    "The assessment uses multiple content indicators together; "
    "no single factor alone determines whether an article is limited."
)

             WIKIWEAK ARTICLE ANALYSIS
Article: Arjun Ray
Character Count: 99
Sections: 1
Images: 0
Infobox Fields: 0
References: 0
Content-Richness Score: 23.88

Assessment:
This article has relatively limited measured content based on the combined analysis of article length, sections, images, infobox information, and references.

Note:
The assessment uses multiple content indicators together; no single factor alone determines whether an article is limited.
